In [1]:
import numpy as np

In [2]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [3]:
def sigmoid_derivative(x):
    s=sigmoid(x)
    return s * (1-s)

In [4]:
def tanh(x):
    return np.tanh(x)

In [21]:
def tanh_derivative(x):
    return 1-(np.tanh(x)**2)

In [29]:
class LSTMCell:
    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        self.concat_dim = input_dim + hidden_dim
        self.W_f=np.random.randn(hidden_dim,self.concat_dim) * 0.1
        self.b_f = np.zeros((hidden_dim, 1))
        self.W_i=np.random.randn(hidden_dim,self.concat_dim) * 0.1
        self.b_i = np.zeros((hidden_dim, 1))
        self.W_c=np.random.randn(hidden_dim,self.concat_dim) * 0.1
        self.b_c = np.zeros((hidden_dim, 1))
        self.W_o=np.random.randn(hidden_dim,self.concat_dim) * 0.1
        self.b_o = np.zeros((hidden_dim, 1))
    
    def forward(self,x_t, h_prev, C_prev):
        z = np.vstack((h_prev, x_t))
        
        f_t = sigmoid(np.dot(self.W_f,z) + self.b_f)
        i_t = sigmoid(np.dot(self.W_i,z) + self.b_i)
        c_tilde = tanh(np.dot(self.W_c,z) + self.b_c)
        o_t = sigmoid(np.dot(self.W_o,z) + self.b_o)
        
        C_t = (f_t * C_prev) + (i_t * c_tilde)
        h_t = o_t * tanh(C_t)
        
        return h_t, C_t, (x_t, h_prev, C_prev, z, f_t, i_t, c_tilde, o_t, C_t)
        
    def forward_sequence(self, X):
        self.caches = []
        
        h_t = np.zeros((self.hidden_dim, 1))
        C_t = np.zeros((self.hidden_dim, 1))
        
        hidden_states = []
        
        
        for i in range(X.shape[1]):
            h_t, C_t, cache = self.forward(X[:, i].reshape(-1,1), h_t, C_t)
            hidden_states.append(h_t)
            self.caches.append(cache)
        
        return np.hstack(hidden_states)
    
    def backward_sequence(self, dh_layer):
        dW_f = np.zeros((self.hidden_dim, self.concat_dim))
        dW_i = np.zeros((self.hidden_dim, self.concat_dim))
        dW_c = np.zeros((self.hidden_dim, self.concat_dim))
        dW_o = np.zeros((self.hidden_dim, self.concat_dim))
        
        db_f = np.zeros((self.hidden_dim, 1))
        db_i = np.zeros((self.hidden_dim, 1))
        db_c = np.zeros((self.hidden_dim, 1))
        db_o = np.zeros((self.hidden_dim, 1))
        
        dh_next = np.zeros((self.hidden_dim, 1))
        dC_next = np.zeros((self.hidden_dim, 1))
        
        for t in reversed(range(len(self.caches))):
            x_t, h_prev, C_prev, z, f_t, i_t, c_tilde, o_t, C_t = self.caches[t]
            
            dh_from_layer = dh_layer[:, t].reshape(-1, 1)
            dh = dh_from_layer + dh_next
            do_t = dh * np.tanh(C_t)
            
            dC_spatial = dh * o_t * tanh_derivative(C_t)
            dC = dC_spatial + dC_next
            df_t = dC * C_prev
            di_t = dC * c_tilde
            dc_tilde = dC * i_t
            
            df_raw = df_t * sigmoid_derivative(f_t)
            di_raw = di_t * sigmoid_derivative(i_t)
            dc_raw = dc_tilde * tanh_derivative(c_tilde)
            do_raw = do_t * sigmoid_derivative(o_t)
            
            dW_f += np.dot(df_raw, z.T)
            db_f += df_raw
            dW_i += np.dot(di_raw, z.T)
            db_i += di_raw
            dW_c += np.dot(dc_raw, z.T)
            db_c += dc_raw
            dW_o += np.dot(do_raw, z.T)
            db_o += do_raw
            
            dC_next = dC * f_t
            dz = (np.dot(self.W_f.T, df_raw) + 
            np.dot(self.W_i.T, di_raw) + 
            np.dot(self.W_c.T, dc_raw) + 
            np.dot(self.W_o.T, do_raw))
            dh_next = dz[:self.hidden_dim, :]
            
            return {
            "dW_f": dW_f, "db_f": db_f,
            "dW_i": dW_i, "db_i": db_i,
            "dW_c": dW_c, "db_c": db_c,
            "dW_o": dW_o, "db_o": db_o
            }
            
    def update_parameters(self, grads, learning_rate=0.01):
        self.W_f -= learning_rate * grads["dW_f"]
        self.b_f -= learning_rate * grads["db_f"]
        
        self.W_i -= learning_rate * grads["dW_i"]
        self.b_i -= learning_rate * grads["db_i"]
        
        self.W_c -= learning_rate * grads["dW_c"]
        self.b_c -= learning_rate * grads["db_c"]
        
        self.W_o -= learning_rate * grads["dW_o"]
        self.b_o -= learning_rate * grads["db_o"]
        return
        
    
        


                        
        
    
        
        
        
        
    

In [24]:
# --- Verification Test Drive ---
if __name__ == "__main__":
    # 1. Define network parameters
    INPUT_FEATURES = 3   # e.g., three input channels/features per timestep
    HIDDEN_MEMORY = 4    # e.g., the cell tracks 4 internal states
    SEQUENCE_LENGTH = 5  # e.g., a sequence tracking 5 consecutive timesteps
    
    print("Initializing LSTM cell...")
    lstm = LSTMCell(input_dim=INPUT_FEATURES, hidden_dim=HIDDEN_MEMORY)
    
    # 2. Generate synthetic data matrix of shape (input_dim, sequence_length)
    # This generates random floating numbers from a normal distribution
    synthetic_data = np.random.randn(INPUT_FEATURES, SEQUENCE_LENGTH)
    
    print("\n--- Input Data ---")
    print(f"Synthetic Input Shape: {synthetic_data.shape} (Features, Timesteps)")
    print("Raw Input Array:")
    print(synthetic_data)
    
    # 3. Run the sequence pass
    print("\nProcessing full sequence through LSTM layer...")
    output_history = lstm.forward_sequence(synthetic_data)
    
    print("\n--- Output Results ---")
    print(f"Output Matrix Shape: {output_history.shape} (Hidden Units, Timesteps)")
    print("Resulting Hidden States across time:")
    print(output_history)

Initializing LSTM cell...

--- Input Data ---
Synthetic Input Shape: (3, 5) (Features, Timesteps)
Raw Input Array:
[[-1.10386414  0.16882954 -1.44057024  1.72675191 -0.76772101]
 [-0.72493678 -1.56916204  1.44043153  0.61844366 -0.5898692 ]
 [-1.06185411 -1.1715105   1.15470132 -0.48264619 -0.61974763]]

Processing full sequence through LSTM layer...

--- Output Results ---
Output Matrix Shape: (4, 5) (Hidden Units, Timesteps)
Resulting Hidden States across time:
[[0.32885428 0.27610462 0.25325101 0.25391739 0.21590411]
 [0.39061319 0.29289882 0.2967005  0.19551293 0.21193135]
 [0.41084894 0.30424375 0.2581665  0.28803797 0.2409397 ]
 [0.41017502 0.27400934 0.25731917 0.2270056  0.23201979]]


In [25]:
# Predicting a sine wave 
def compute_mse_loss(predictions, targets):
    """Calculates Mean Squared Error."""
    return np.mean((predictions - targets) ** 2)

def compute_loss_gradient(predictions, targets):
    """Computes the direct derivative of MSE loss with respect to predictions."""
    # This matches the shape of predictions: (hidden_dim, sequence_length)
    return 2 * (predictions - targets) / predictions.size


In [31]:
if __name__ == "__main__":
    np.random.seed(42) # For reproducible results
    
    # 1. Setup dimensions
    INPUT_FEATURES = 1   # 1 value per timestep (the current point on the wave)
    HIDDEN_MEMORY = 8    # 1 hidden unit, which acts directly as our prediction
    SEQUENCE_LENGTH = 10 # Look at 10 points in time to predict
    
    # Initialize your complete cell
    lstm = LSTMCell(input_dim=INPUT_FEATURES, hidden_dim=HIDDEN_MEMORY)
    W_out = np.random.randn(1, HIDDEN_MEMORY) * 0.1
    b_out = np.zeros((1, 1))
    
    # 2. Create Synthetic Sine Wave Data
    # We generate a long sine wave and slice it into input sequences and target outputs
    steps = np.linspace(0, 50, 200)
    sine_wave = np.sin(steps).reshape(1, -1) # Shape: (1, 200)
    
    # Extract one sample training sequence of 10 steps
    # X_train: timesteps 0 to 9
    X_train = sine_wave[:, :SEQUENCE_LENGTH] 
    # Y_train: timesteps 1 to 10 (the network tries to predict the next step ahead)
    Y_train = sine_wave[:, 1:SEQUENCE_LENGTH + 1] 
    
    print("--- Training Data Ready ---")
    print(f"X_train Shape: {X_train.shape} | Y_train Shape: {Y_train.shape}")
    
    # 3. The Optimization Loop
    EPOCHS = 400
    LEARNING_RATE = 0.1
    
    print(f"\nStarting training for {EPOCHS} epochs...")
    print("-" * 40)
    
    for epoch in range(1, EPOCHS + 1):
        # 1. Forward Pass through LSTM
        hidden_history = lstm.forward_sequence(X_train) # Shape: (8, 10)
        
        # 2. Linear Projection Layer to get 1D predictions
        predictions = np.dot(W_out, hidden_history) + b_out # Shape: (1, 10)
        
        # 3. Compute Loss
        loss = compute_mse_loss(predictions, Y_train)
        
        # 4. Backpropagate through the Linear Projection Layer
        dpredictions = compute_loss_gradient(predictions, Y_train) # Shape: (1, 10)
        dW_out = np.dot(dpredictions, hidden_history.T)
        db_out = np.sum(dpredictions, axis=1, keepdims=True)
        
        # 5. Calculate dh_layer to pass backward into the LSTM cell
        dh_layer = np.dot(W_out.T, dpredictions) # Shape: (8, 10)
        
        # 6. Backward Pass through LSTM
        gradients = lstm.backward_sequence(dh_layer)
        
        # 7. Update ALL Parameters (LSTM + Output Layer)
        lstm.update_parameters(gradients, learning_rate=LEARNING_RATE)
        W_out -= LEARNING_RATE * dW_out
        b_out -= LEARNING_RATE * db_out
        
        if epoch == 1 or epoch % 40 == 0:
            print(f"Epoch {epoch:3d} | Mean Squared Error Loss: {loss:.6f}")
            
    print("-" * 40)
    print("Training Complete!")

--- Training Data Ready ---
X_train Shape: (1, 10) | Y_train Shape: (1, 10)

Starting training for 400 epochs...
----------------------------------------
Epoch   1 | Mean Squared Error Loss: 0.658894
Epoch  40 | Mean Squared Error Loss: 0.063247
Epoch  80 | Mean Squared Error Loss: 0.053013
Epoch 120 | Mean Squared Error Loss: 0.044955
Epoch 160 | Mean Squared Error Loss: 0.038595
Epoch 200 | Mean Squared Error Loss: 0.033564
Epoch 240 | Mean Squared Error Loss: 0.029577
Epoch 280 | Mean Squared Error Loss: 0.026415
Epoch 320 | Mean Squared Error Loss: 0.023908
Epoch 360 | Mean Squared Error Loss: 0.021924
Epoch 400 | Mean Squared Error Loss: 0.020359
----------------------------------------
Training Complete!
